# 分布式数据并行实操

分布式数据并行（DDP）是 PyTorch 中一个强大的模块，它允许您在多个机器上并行化您的模型，非常适合大规模深度学习应用。要使用 DDP，您需要生成多个进程，并为每个进程创建一个 DDP 的单一实例。  
但是它是如何工作的呢？DDP 使用 torch.distributed 包中的集体通信来同步所有进程的梯度和缓冲区。这意味着每个进程将拥有模型的自己的副本，但它们将共同工作以训练模型，就像在一台机器上一样。

为了实现这一点，DDP 为模型中的每个参数注册了一个自动求导钩子。当执行反向传播时，这个钩子会触发并在所有进程之间同步梯度。这确保每个进程具有相同的梯度，然后用于更新模型。  
有关 DDP 如何工作以及如何有效使用它的更多信息，请务必查看 DDP 设计说明。使用 DDP，您可以比以往更快、更高效地训练模型！

使用 DDP 的推荐方法是为每个模型副本生成一个进程。模型副本可以跨多个设备。DDP 进程可以放置在同一台机器上或跨机器。请注意，GPU 设备不能在 DDP 进程之间共享（即一个 GPU 对应一个 DDP 进程）。

在本教程中，我们将从一个基本的 DDP 用例开始，然后演示更高级的用例，包括模型检查点和将 DDP 与模型并行结合。



## 简单Example示例
让我们从一个简单的 `torch.nn.parallel.DistributedDataParallel` 示例开始。这个示例使用 `torch.nn.Linear` 作为本地模型，使用 `DDP` 包装它，然后在 `DDP` 模型上运行一次前向传播、一次反向传播和一次优化器步骤。之后，本地模型上的参数将被更新，所有不同进程上的模型应该完全相同。

In [ ]:
import torch
import torch.distributed as dist
import torch.multiprocessing as mp
import torch.nn as nn
import torch.optim as optim
import os
from torch.nn.parallel import DistributedDataParallel as DDP


def example(rank, world_size):
    # create default process group
    dist.init_process_group("gloo", rank=rank, world_size=world_size)
    # create local model
    model = nn.Linear(10, 10).to(rank)
    # construct DDP model
    ddp_model = DDP(model, device_ids=[rank])
    # define loss function and optimizer
    loss_fn = nn.MSELoss()
    optimizer = optim.SGD(ddp_model.parameters(), lr=0.001)

    # forward pass
    outputs = ddp_model(torch.randn(20, 10).to(rank))
    labels = torch.randn(20, 10).to(rank)
    # backward pass
    loss_fn(outputs, labels).backward()
    # update parameters
    optimizer.step()

def main():
    world_size = 2
    mp.spawn(example,
        args=(world_size,),
        nprocs=world_size,
        join=True)

if __name__=="__main__":
    # Environment variables which need to be
    # set when using c10d's default "env"
    # initialization mode.
    os.environ["MASTER_ADDR"] = "localhost"
    os.environ["MASTER_PORT"] = "29500"
    main()

## DDP工作原理回顾
### 初始化
DDP 构造函数引用本地模块，并从秩为 0 的进程向组中的所有其他进程广播 state_dict() ，以确保所有模型副本从完全相同的状态开始。然后，每个 DDP 进程创建一个本地 Reducer ，它将在反向传播期间负责梯度同步。为了提高通信效率， Reducer 将参数梯度组织成桶，并一次减少一个桶。桶大小可以通过在 DDP 构造函数中设置 bucket_cap_mb 参数进行配置。参数梯度到桶的映射在构造时确定，基于桶大小限制和参数大小。模型参数按给定模型的 Model.parameters() 的（大致）反向顺序分配到桶中。使用反向顺序的原因是因为 DDP 期望梯度在反向传播期间大致按该顺序准备就绪。下图显示了一个示例。请注意， grad0 和 grad1 在 bucket1 中，其他两个梯度在 bucket0 中。 当然，这个假设可能并不总是正确的，当这种情况发生时，可能会影响 DDP 的反向速度，因为 Reducer 无法在最早的时间启动通信。除了分桶之外， Reducer 在构造期间还会为每个参数注册自动梯度钩子。这些钩子将在反向传播时触发，当梯度准备好时。
### 前向传播
DDP 接收输入并将其传递给本地模型，然后分析本地模型的输出，如果 find_unused_parameters 设置为 True 。此模式允许在模型的子图上进行反向传播，DDP 通过从模型输出遍历自动求导图并标记所有未使用的参数为准备好进行归约，找出参与反向传播的参数。在反向传播过程中， Reducer 只会等待未准备好的参数，但仍会减少所有桶。将参数梯度标记为准备好并不会帮助 DDP 跳过桶，但它将防止 DDP 在反向传播期间永远等待缺失的梯度。请注意，遍历自动求导图会引入额外的开销，因此应用程序应仅在必要时将 find_unused_parameters 设置为 True 。
### 反向传播
backward() 函数直接在损失 Tensor 上调用，这超出了 DDP 的控制，DDP 使用在构造时注册的自动梯度钩子来触发梯度同步。当一个梯度准备好时，它对应的 DDP 钩子将在该梯度累加器上触发，然后 DDP 将标记该参数梯度为准备好进行归约。当一个桶中的所有梯度都准备好时， Reducer 将启动一个异步 allreduce 来计算所有进程的梯度均值。当所有桶都准备好时， Reducer 将阻塞等待所有 allreduce 操作完成。当完成后，平均梯度将写入所有参数的 param.grad 字段。因此，在反向传播后，不同 DDP 进程中相同对应参数的梯度字段应该是相同的。
### 优化器
从优化器的角度来看，它正在优化一个本地模型。所有 DDP 进程上的模型副本可以保持同步，因为它们都从相同的状态开始，并且在每次迭代中具有相同的平均梯度。

![alt text](../_img/DDP_M.png)

#### 注意：
DDP 要求所有进程上有 Reducer 个实例以完全相同的顺序调用 allreduce ，这通过始终按照桶索引顺序而不是实际桶就绪顺序运行 allreduce 来实现。进程之间不匹配的 allreduce 顺序可能导错误结果或 DDP 后向挂起。

## DataParallel 和 DistributedDataParallel 之间的比较
在我们深入之前，让我们澄清一下为什么您会考虑使用 DistributedDataParallel 而不是 DataParallel ，尽管它增加了复杂性：
- 首先， DataParallel 是单进程、多线程的，但它仅在单台机器上工作。相比之下， DistributedDataParallel 是多进程的，支持单机和多机训练。由于线程之间的 GIL 争用、每次迭代的模型复制以及输入分散和输出聚合引入的额外开销，即使在单台机器上， DataParallel 通常也比 DistributedDataParallel 慢。
- 回想一下之前的教程，如果您的模型太大而无法放入单个 GPU，您必须使用模型并行将其分割到多个 GPU 上。 DistributedDataParallel 可以与模型并行一起使用，而 DataParallel 目前不能。当 DDP 与模型并行结合时，每个 DDP 进程将使用模型并行，所有进程将共同使用数据并行。

In [ ]:
import os
import sys
import tempfile
import torch
import torch.distributed as dist
import torch.nn as nn
import torch.optim as optim
import torch.multiprocessing as mp

from torch.nn.parallel import DistributedDataParallel as DDP

# On Windows platform, the torch.distributed package only
# supports Gloo backend, FileStore and TcpStore.
# For FileStore, set init_method parameter in init_process_group
# to a local file. Example as follow:
# init_method="file:///f:/libtmp/some_file"
# dist.init_process_group(
#    "gloo",
#    rank=rank,
#    init_method=init_method,
#    world_size=world_size)
# For TcpStore, same way as on Linux.

def setup(rank, world_size):
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'

    # initialize the process group
    dist.init_process_group("gloo", rank=rank, world_size=world_size)

def cleanup():
    dist.destroy_process_group()

现在，让我们创建一个Toy模块，用 DDP 封装它，并为它提供一些虚拟输入数据。请注意，由于 DDP 在 DDP 构造函数中将模型状态从 rank 0 进程广播到所有其他进程，因此您无需担心不同 DDP 进程从不同的初始模型参数值开始。

In [ ]:
class ToyModel(nn.Module):
    def __init__(self):
        super(ToyModel, self).__init__()
        self.net1 = nn.Linear(10, 10)
        self.relu = nn.ReLU()
        self.net2 = nn.Linear(10, 5)

    def forward(self, x):
        return self.net2(self.relu(self.net1(x)))


def demo_basic(rank, world_size):
    print(f"Running basic DDP example on rank {rank}.")
    setup(rank, world_size)

    # create model and move it to GPU with id rank
    model = ToyModel().to(rank)
    ddp_model = DDP(model, device_ids=[rank])

    loss_fn = nn.MSELoss()
    optimizer = optim.SGD(ddp_model.parameters(), lr=0.001)

    optimizer.zero_grad()
    outputs = ddp_model(torch.randn(20, 10))
    labels = torch.randn(20, 5).to(rank)
    loss_fn(outputs, labels).backward()
    optimizer.step()

    cleanup()
    print(f"Finished running basic DDP example on rank {rank}.")


def run_demo(demo_fn, world_size):
    mp.spawn(demo_fn,
             args=(world_size,),
             nprocs=world_size,
             join=True)

如您所见，DDP 封装了低级分布式通信细节，并提供了一个干净的 API，就像它是一个本地模型一样。梯度同步通信发生在反向传播期间，并与反向计算重叠。当 backward() 返回时， param.grad 已经包含了同步的梯度张量。对于基本用例，DDP 只需要多几行代码来设置进程组。当将 DDP 应用于更高级的用例时，一些注意事项需要谨慎对待。


### Skewed Processing Speeds  倾斜的处理速度
在 DDP 中，构造函数、前向传播和反向传播是分布式同步点。不同的进程预计会启动相同数量的同步并以相同的顺序到达这些同步点，并大致在同一时间进入每个同步点。否则，快速进程可能会提前到达并在等待滞后进程时超时。因此，用户有责任在进程之间平衡工作负载分配。有时，由于网络延迟、资源竞争或不可预测的工作负载峰值，处理速度的不均衡是不可避免的。为了避免在这些情况下超时，请确保在调用 init_process_group 时传递一个足够大的 timeout 值。
## Save and Load Checkpoints
### 保存和加载检查点
在训练期间，使用 torch.save 和 torch.load 来检查点模块并从检查点恢复是很常见的。有关更多详细信息，请参见保存和加载模型。当使用 DDP 时，一种优化是仅在一个进程中保存模型，然后在所有进程中加载，从而减少写入开销。这是可行的，因为所有进程都从相同的参数开始，并且在反向传播中梯度是同步的，因此优化器应该保持将参数设置为相同的值。如果您使用此优化（即在一个进程中保存但在所有进程中恢复），请确保在保存完成之前没有进程开始加载。此外，在加载模块时，您需要提供适当的 map_location 参数，以防止进程进入其他进程的设备。如果 map_location 缺失， torch.load 将首先将模块加载到 CPU，然后将每个参数复制到保存的位置，这将导致同一台机器上的所有进程使用相同的设备集。有关更高级的故障恢复和弹性支持，请参阅 TorchElastic。

In [ ]:
def demo_checkpoint(rank, world_size):
    print(f"Running DDP checkpoint example on rank {rank}.")
    setup(rank, world_size)

    model = ToyModel().to(rank)
    ddp_model = DDP(model, device_ids=[rank])


    CHECKPOINT_PATH = tempfile.gettempdir() + "/model.checkpoint"
    if rank == 0:
        # All processes should see same parameters as they all start from same
        # random parameters and gradients are synchronized in backward passes.
        # Therefore, saving it in one process is sufficient.
        torch.save(ddp_model.state_dict(), CHECKPOINT_PATH)

    # Use a barrier() to make sure that process 1 loads the model after process
    # 0 saves it.
    dist.barrier()
    # configure map_location properly
    map_location = {'cuda:%d' % 0: 'cuda:%d' % rank}
    ddp_model.load_state_dict(
        torch.load(CHECKPOINT_PATH, map_location=map_location, weights_only=True))

    loss_fn = nn.MSELoss()
    optimizer = optim.SGD(ddp_model.parameters(), lr=0.001)

    optimizer.zero_grad()
    outputs = ddp_model(torch.randn(20, 10))
    labels = torch.randn(20, 5).to(rank)

    loss_fn(outputs, labels).backward()
    optimizer.step()

    # Not necessary to use a dist.barrier() to guard the file deletion below
    # as the AllReduce ops in the backward pass of DDP already served as
    # a synchronization.

    if rank == 0:
        os.remove(CHECKPOINT_PATH)

    cleanup()
    print(f"Finished running DDP checkpoint example on rank {rank}.")

## 结合 DDP 与模型并行性
DDP 也适用于多 GPU 模型。DDP 包装多 GPU 模型在训练大型模型时尤其有用，因为它可以处理大量数据

In [ ]:
class ToyMpModel(nn.Module):
    def __init__(self, dev0, dev1):
        super(ToyMpModel, self).__init__()
        self.dev0 = dev0
        self.dev1 = dev1
        self.net1 = torch.nn.Linear(10, 10).to(dev0)
        self.relu = torch.nn.ReLU()
        self.net2 = torch.nn.Linear(10, 5).to(dev1)

    def forward(self, x):
        x = x.to(self.dev0)
        x = self.relu(self.net1(x))
        x = x.to(self.dev1)
        return self.net2(x)

在将多 GPU 模型传递给 DDP 时， device_ids 和 output_device 必须不设置。输入和输出数据将由应用程序或模型 forward() 方法放置在适当的设备上。

In [ ]:
def demo_model_parallel(rank, world_size):
    print(f"Running DDP with model parallel example on rank {rank}.")
    setup(rank, world_size)

    # setup mp_model and devices for this process
    dev0 = rank * 2
    dev1 = rank * 2 + 1
    mp_model = ToyMpModel(dev0, dev1)
    ddp_mp_model = DDP(mp_model)

    loss_fn = nn.MSELoss()
    optimizer = optim.SGD(ddp_mp_model.parameters(), lr=0.001)

    optimizer.zero_grad()
    # outputs will be on dev1
    outputs = ddp_mp_model(torch.randn(20, 10))
    labels = torch.randn(20, 5).to(dev1)
    loss_fn(outputs, labels).backward()
    optimizer.step()

    cleanup()
    print(f"Finished running DDP with model parallel example on rank {rank}.")


if __name__ == "__main__":
    n_gpus = torch.cuda.device_count()
    assert n_gpus >= 2, f"Requires at least 2 GPUs to run, but got {n_gpus}"
    world_size = n_gpus
    run_demo(demo_basic, world_size)
    run_demo(demo_checkpoint, world_size)
    world_size = n_gpus//2
    run_demo(demo_model_parallel, world_size)

### 使用 torch.distributed.run/torchrun 初始化 DDP
我们可以利用 PyTorch Elastic 来简化 DDP 代码，并更轻松地初始化作业。我们仍然使用 Toymodel 示例，并创建一个名为 elastic_ddp.py 的文件。

In [ ]:
import torch
import torch.distributed as dist
import torch.nn as nn
import torch.optim as optim

from torch.nn.parallel import DistributedDataParallel as DDP

class ToyModel(nn.Module):
    def __init__(self):
        super(ToyModel, self).__init__()
        self.net1 = nn.Linear(10, 10)
        self.relu = nn.ReLU()
        self.net2 = nn.Linear(10, 5)

    def forward(self, x):
        return self.net2(self.relu(self.net1(x)))


def demo_basic():
    torch.cuda.set_device(int(os.environ["LOCAL_RANK"]))
    dist.init_process_group("nccl")
    rank = dist.get_rank()
    print(f"Start running basic DDP example on rank {rank}.")
    # create model and move it to GPU with id rank
    device_id = rank % torch.cuda.device_count()
    model = ToyModel().to(device_id)
    ddp_model = DDP(model, device_ids=[device_id])
    loss_fn = nn.MSELoss()
    optimizer = optim.SGD(ddp_model.parameters(), lr=0.001)

    optimizer.zero_grad()
    outputs = ddp_model(torch.randn(20, 10))
    labels = torch.randn(20, 5).to(device_id)
    loss_fn(outputs, labels).backward()
    optimizer.step()
    dist.destroy_process_group()
    print(f"Finished running basic DDP example on rank {rank}.")

if __name__ == "__main__":
    demo_basic()

可以在所有节点上运行 torch elastic/torchrun 命令来初始化上述创建的 DDP 作业：

In [ ]:
!torchrun --nnodes=2 --nproc_per_node=8 --rdzv_id=100 --rdzv_backend=c10d --rdzv_endpoint=$MASTER_ADDR:29400 elastic_ddp.py

在上面的示例中，我们在两个主机上运行 DDP 脚本，并且在每个主机上运行 8 个进程。也就是说，我们在 16 个 GPU 上运行此作业。请注意， $MASTER_ADDR 在所有节点上必须相同。
这里 torchrun 将启动 8 个进程，并在其启动的节点上对每个进程调用 elastic_ddp.py ，但用户还需要应用集群管理工具，如 slurm，才能在 2 个节点上实际运行此命令。
例如，在启用 SLURM 的集群上，我们可以编写一个脚本来运行上述命令并将 MASTER_ADDR 设置为：

In [ ]:
!export MASTER_ADDR=$(scontrol show hostname ${SLURM_NODELIST} | head -n 1)

然后我们可以使用 SLURM 命令运行这个脚本： srun --nodes=2 ./torchrun_script.sh 。
这是一个示例；您可以选择自己的集群调度工具来启动 torchrun 作业。